## Dataset analysis: `student_high_schools_all.csv`

Purpose: student → high school linkage, including tier/ownership and `STUDENT_LIST` cohort label.

Used for enrichment features (school tier/ownership/district) and potential cohort splits (List15 vs List16).

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "student_high_schools_all.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.isna().mean().sort_values(ascending=False)

In [ ]:
key = ["REG_NO"]
df.duplicated(key).sum(), df[key].isna().any(axis=1).sum()

In [ ]:
df["STUDENT_LIST"].value_counts(dropna=False)

In [ ]:
df["SCHOOL_TIER"].value_counts(dropna=False)

In [ ]:
df["OWNERSHIP"].value_counts(dropna=False).head(20)

## Advanced analytics

Focus: CGPA by school tier, ownership, and district (via transcript join).

In [ ]:
from analysis_utils import basic_profile, missingness_report

print(basic_profile(df))
missingness_report(df, top_n=20)

In [ ]:
# Join to combined transcript for CGPA by school tier/ownership/district
trans_all = pd.concat([
    pd.read_csv(DATA_DIR / "student_transcript_list15.csv"),
    pd.read_csv(DATA_DIR / "student_transcript_list16.csv"),
], ignore_index=True)

trans_all["CGPA"] = pd.to_numeric(trans_all["CGPA"], errors="coerce")

merged = df.merge(trans_all[["REG_NO","CGPA"]], on="REG_NO", how="left")

print("CGPA by SCHOOL_TIER:")
print(merged.groupby("SCHOOL_TIER")["CGPA"].agg(["count","mean","median","std"]).sort_values("mean", ascending=False))

print("\nCGPA by OWNERSHIP:")
print(merged.groupby("OWNERSHIP")["CGPA"].agg(["count","mean","median","std"]).sort_values("mean", ascending=False))